# What this notebook is, and what it found

**Purpose:** the Milestone 3 report (\u00a75.2/\u00a77.2) credits `BAAI/bge-m3` with a
strong ability to separate agriculture topics \u2014 but that number was
measured on a **teammate's dataset**, not KCC. `05_kcc_embedding_indexing.ipynb`
and `06_kcc_retrieval_eval.ipynb` had already shown MuRIL fails and LaBSE
gives only a modest improvement on KCC specifically. This notebook checks
whether bge-m3's reported strength actually holds on **our own KCC data**,
using a smaller/faster version of the same methodology so it doesn't take
an hour to find out.

**Method:** 1,000 stratified KCC chunks, embedded with bge-m3; same Cohen's
d anisotropy check (Category and QueryType), a held-out generalization eval
(100 queries, category/crop-match@5 vs. random), and the same 10
qualitative farmer-style test queries used for MuRIL/LaBSE.

## Results (this run)

| Metric | bge-m3 | For comparison: MuRIL | For comparison: LaBSE |
|---|---|---|---|
| Cohen's d (Category) | 0.315 (small) | \u22120.001 | 0.047 |
| Cohen's d (QueryType) | 0.658 (moderate) | 0.722* | 0.719 |
| Held-out Category-match@5 | **89.0%** | 67.0% | 69.0% |
| Held-out Crop-match@5 | **81.0%** | 33.0% | 42.0% |
| Crop-match lift over random | **+62.0%** | +6.5% | +15.5% |
| Random baseline (this run) | 19.0% crop-match | 26.5%\u2020 | 26.5%\u2020 |

*MuRIL's QueryType d looks comparable to bge-m3's but is an artifact of
near-zero variance around a tiny absolute gap \u2014 see `05` for the full
explanation. \u2020 Random baseline differs slightly across notebooks because
this run used a smaller 1,000-chunk index than `06`'s 5,000-chunk index;
the *within-notebook* model-vs-random comparison is what matters, not the
baseline value across notebooks.

**Bottom line:** bge-m3 is dramatically better than both MuRIL and LaBSE
on KCC data specifically \u2014 not a marginal win, a different tier of
retrieval quality (roughly 4x LaBSE's and 10x MuRIL's lift over random).
Qualitative spot-checks confirm it: a Hindi query about wheat yellow
disease correctly retrieved wheat yellow-rust treatment at top-1, something
LaBSE missed entirely. The one consistent miss across all three
models \u2014 government scheme queries (e.g. "PM Kisan eligibility") \u2014 turned
out to be a **KCC corpus gap** (no scheme content exists in KCC alone), not
a model failure; the team's production index resolves this by including a
separate PDF policy corpus alongside KCC (see the RAG production findings
summary).

**Relationship to the Milestone 3 report:** this independently confirms,
on KCC specifically, the same model choice already made and validated at
much larger scale in the team's production RAG build (\u00a75.2, 723,439-chunk
index). This notebook is corroborating evidence, not a competing
justification \u2014 bge-m3 was already the right call before this ran.


# KCC \u2014 Quick BGE-M3 Test

**Why this notebook:** the M3 report (v13, \u00a77.2) cites specific
anisotropy numbers for `BAAI/bge-m3` (median cosine 0.404 for unrelated
chunks) that were reportedly measured on a **teammate's dataset**, not
KCC. Before trusting that number for KCC too, this notebook runs the
same measurement \u2014 Cohen's d anisotropy diagnostic + held-out
generalization eval \u2014 on your own KCC chunks, fast.

**Kept deliberately lightweight so it actually finishes quickly:**
- Smaller sample (1,000 chunks, not 5,000) \u2014 still stratified by
  Category \u00d7 Crop so it's representative, just smaller.
- 300 diagnostic pairs per group (not 500).
- 100 held-out eval queries (not 200).
- Only bge-m3 is embedded here \u2014 MuRIL/LaBSE are NOT re-run. Their
  numbers from your last full run are hardcoded below as a reference for
  the comparison table (update these two dicts if your saved
  `kcc_embedding_comparison_summary.json` on Drive has different values).

**Output:** one comparison table (MuRIL vs LaBSE vs bge-m3, on YOUR
KCC data, same methodology) \u2014 which either supports or corrects the
number currently sitting in \u00a77.2 of the M3 report.


In [1]:
# Step 0: Install dependencies (uncomment on a fresh Colab runtime)
!pip install -q sentence-transformers faiss-cpu transformers torch


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 71.3 MB/s eta 0:00:00


In [2]:
# Code using mounted Google Drive (commented out for colleague's convenience; uncomment to run on Colab)
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [3]:
import json
import time
import numpy as np
import pandas as pd
from pathlib import Path
from collections import Counter

import torch
from sentence_transformers import SentenceTransformer
import faiss

import warnings
warnings.filterwarnings('ignore')

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")


Using device: cpu


## Step 1: Load a Small, Fast Stratified Sample

Reuses the same stratified-by-(Category, Crop) logic from
`04_kcc_preprocessing.ipynb`, just at a smaller size so this runs in
minutes, not tens of minutes.

In [4]:
BASE_PATH = "/content/drive/MyDrive/kcc_raw/"
PROCESSED_PATH = f"{BASE_PATH}processed/"
FINAL_PATH = f"{BASE_PATH}final/"

# Prefer sampling straight from the full corpus if available (more honest
# than re-sampling from the already-sampled 5000-chunk file); fall back to
# the 5000-chunk file if the full corpus isn't handy on this run.
FULL_CHUNKS_PATH = f"{PROCESSED_PATH}kcc_chunks_rag.jsonl"
FALLBACK_CHUNKS_PATH = f"{PROCESSED_PATH}kcc_chunks_sample_5000.jsonl"

source_path = FULL_CHUNKS_PATH if Path(FULL_CHUNKS_PATH).exists() else FALLBACK_CHUNKS_PATH
if not Path(source_path).exists():
    raise FileNotFoundError(
        f"\u274c Neither '{FULL_CHUNKS_PATH}' nor '{FALLBACK_CHUNKS_PATH}' found. "
        "Run 04_kcc_preprocessing.ipynb first."
    )
print(f"Sampling from: {source_path}")

all_chunks = []
with open(source_path, 'r', encoding='utf-8') as f:
    for line in f:
        all_chunks.append(json.loads(line))
print(f"Source pool: {len(all_chunks):,} chunks")


def stratified_chunk_sample(chunks, n_total, random_state=7):
    rng = np.random.default_rng(random_state)
    buckets = {}
    for i, c in enumerate(chunks):
        key = (c['metadata'].get('category', 'unknown'), c['metadata'].get('crop', 'unknown'))
        buckets.setdefault(key, []).append(i)
    total = len(chunks)
    sampled_idx = []
    for key, idxs in buckets.items():
        share = len(idxs) / total
        n_take = max(1, round(share * n_total))
        n_take = min(n_take, len(idxs))
        sampled_idx.extend(rng.choice(idxs, size=n_take, replace=False).tolist())
    rng.shuffle(sampled_idx)
    return [chunks[i] for i in sorted(sampled_idx[:n_total])]


N_QUICK = 1000
chunks = stratified_chunk_sample(all_chunks, N_QUICK, random_state=7)
texts = [c['text'] for c in chunks]
meta = [c['metadata'] for c in chunks]
print(f"Quick-test sample: {len(chunks):,} chunks (stratified by Category x Crop)")


Sampling from: /content/drive/MyDrive/kcc_raw/processed/kcc_chunks_sample_5000.jsonl
Source pool: 5,000 chunks
Quick-test sample: 1,000 chunks (stratified by Category x Crop)


## Step 2: Load BGE-M3

`BAAI/bge-m3` outputs 1024-dim dense embeddings (larger than MuRIL/LaBSE's
768) and natively supports normalized similarity search via
`sentence-transformers`. It's also a bigger model than either MuRIL or
LaBSE (XLM-RoBERTa-large backbone) \u2014 expect it to be the slowest of the
three per-chunk, which is exactly why this notebook uses a smaller
sample.

In [5]:
MODEL_NAME = "BAAI/bge-m3"
print(f"Loading {MODEL_NAME} ...")
bge_model = SentenceTransformer(MODEL_NAME, device=DEVICE)
BGE_DIM = bge_model.get_sentence_embedding_dimension()
print(f"\u2705 Loaded. Embedding dim: {BGE_DIM}")


def embed_bge(texts_batch, batch_size=16):
    return bge_model.encode(
        texts_batch, batch_size=batch_size,
        normalize_embeddings=True, show_progress_bar=True
    )


Loading BAAI/bge-m3 ...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/15.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

✅ Loaded. Embedding dim: 1024


## Step 3: Embed the Sample

In [6]:
print(f"Embedding {len(texts):,} chunks with bge-m3...")
t0 = time.time()
bge_embeddings = embed_bge(texts, batch_size=16)
bge_time = time.time() - t0
print(f"\u2705 Done in {bge_time:.1f}s ({len(texts)/bge_time:.1f} chunks/sec)")
print(f"Shape: {bge_embeddings.shape}")

n_nan = np.isnan(bge_embeddings).any(axis=1).sum()
norms = np.linalg.norm(bge_embeddings, axis=1)
print(f"NaNs: {n_nan} | norm mean={norms.mean():.4f} min={norms.min():.4f} max={norms.max():.4f}")
assert n_nan == 0, "NaNs in embeddings \u2014 stop before indexing."


Embedding 1,000 chunks with bge-m3...


Batches:   0%|          | 0/63 [00:00<?, ?it/s]

✅ Done in 551.4s (1.8 chunks/sec)
Shape: (1000, 1024)
NaNs: 0 | norm mean=1.0000 min=1.0000 max=1.0000


## Step 4: Anisotropy Diagnostic (same methodology as MuRIL/LaBSE)

Same 300-pair same-group vs. different-group Cohen's d test as `05`,
grouped by both `category` (coarse, crop-group level) and `query_type`
(finer, actual topic level \u2014 the one that mattered for LaBSE).

In [7]:
def sample_pairs_by_field(field, n_pairs, same_group, random_state):
    rng = np.random.default_rng(random_state)
    pairs = []
    attempts = 0
    while len(pairs) < n_pairs and attempts < n_pairs * 50:
        i, j = rng.integers(0, len(chunks), size=2)
        attempts += 1
        if i == j:
            continue
        is_same = meta[i].get(field, 'unknown') == meta[j].get(field, 'unknown')
        if is_same == same_group:
            pairs.append((i, j))
    return pairs


def cohens_d(a, b):
    pooled_std = np.sqrt((np.std(a, ddof=1)**2 + np.std(b, ddof=1)**2) / 2)
    return (np.mean(a) - np.mean(b)) / pooled_std if pooled_std > 0 else 0.0


def diagnose_field(embeddings, model_name, field_name, n_pairs=300, seed=(42, 43)):
    same_pairs = sample_pairs_by_field(field_name, n_pairs, True, seed[0])
    diff_pairs = sample_pairs_by_field(field_name, n_pairs, False, seed[1])
    same_sims = np.array([np.dot(embeddings[i], embeddings[j]) for i, j in same_pairs])
    diff_sims = np.array([np.dot(embeddings[i], embeddings[j]) for i, j in diff_pairs])
    d = cohens_d(same_sims, diff_sims)

    print(f"\n{model_name}  (grouped by {field_name}, n={len(same_pairs)}/{len(diff_pairs)} pairs)")
    print("-" * 55)
    print(f"  Same-{field_name} similarity:      mean={same_sims.mean():.4f}  std={same_sims.std():.4f}")
    print(f"  Different-{field_name} similarity: mean={diff_sims.mean():.4f}  std={diff_sims.std():.4f}")
    print(f"  Cohen's d:                      {d:.3f}", end="  ")
    print("\u2192 negligible" if d < 0.2 else "\u2192 small" if d < 0.5 else "\u2192 moderate" if d < 0.8 else "\u2192 strong")

    return {"same_mean": float(same_sims.mean()), "same_std": float(same_sims.std()),
            "diff_mean": float(diff_sims.mean()), "diff_std": float(diff_sims.std()), "cohens_d": float(d)}


qtype_dist = Counter(m.get('query_type', 'unknown') for m in meta)
print("QueryType distribution in this 1,000-chunk sample:")
for qt, count in qtype_dist.most_common(8):
    print(f"  {qt:35s} {count:4d}  ({100*count/len(meta):.1f}%)")

bge_diag_cat = diagnose_field(bge_embeddings, "bge-m3", "category")
bge_diag_qtype = diagnose_field(bge_embeddings, "bge-m3", "query_type")


QueryType distribution in this 1,000-chunk sample:
  Plant Protection                     454  (45.4%)
  Nutrient Management                  114  (11.4%)
  Cultural Practices                   113  (11.3%)
  Fertilizer Use and Availability       95  (9.5%)
  Weed Management                       65  (6.5%)
  Varieties                             49  (4.9%)
  Seeds and Planting Material           31  (3.1%)
  Water Management                      15  (1.5%)

bge-m3  (grouped by category, n=300/300 pairs)
-------------------------------------------------------
  Same-category similarity:      mean=0.5672  std=0.1028
  Different-category similarity: mean=0.5375  std=0.0846
  Cohen's d:                      0.315  → small

bge-m3  (grouped by query_type, n=300/300 pairs)
-------------------------------------------------------
  Same-query_type similarity:      mean=0.5830  std=0.0851
  Different-query_type similarity: mean=0.5275  std=0.0833
  Cohen's d:                      0.658  → mode

## Step 5: Comparison Against MuRIL / LaBSE

MuRIL and LaBSE are **not re-embedded here** \u2014 that would defeat the
point of a quick test. Their numbers below are your actual measured
results from the full `05`/`06` run (500 pairs, 5,000-chunk sample).
If you have `kcc_embedding_comparison_summary.json` saved on Drive from
that run, this cell loads it automatically instead of using the
hardcoded fallback \u2014 check the printed source so you know which one
was used.

In [8]:
# Hardcoded fallback = your actual measured numbers from the full 05/06 run
MURIL_REFERENCE = {
    "diagnostic_by_category": {"cohens_d": -0.001},
    "diagnostic_by_query_type": {"cohens_d": 0.722},
}
LABSE_REFERENCE = {
    "diagnostic_by_category": {"cohens_d": 0.047},
    "diagnostic_by_query_type": {"cohens_d": 0.719},
}

prior_summary_path = f"{FINAL_PATH}kcc_embedding_comparison_summary.json"
if Path(prior_summary_path).exists():
    with open(prior_summary_path, 'r', encoding='utf-8') as f:
        prior = json.load(f)
    muril_ref = prior.get("muril", MURIL_REFERENCE)
    labse_ref = prior.get("labse", LABSE_REFERENCE)
    print(f"Loaded prior MuRIL/LaBSE numbers from: {prior_summary_path}")
else:
    muril_ref = MURIL_REFERENCE
    labse_ref = LABSE_REFERENCE
    print("\u26a0\ufe0f  No saved prior summary found \u2014 using hardcoded reference numbers from your last "
          "reported run. Verify these match if you've re-run 05/06 since.")

print(f"\n{'Model':10s} {'d (Category)':>14s} {'d (QueryType)':>15s}")
print(f"{'MuRIL':10s} {muril_ref['diagnostic_by_category']['cohens_d']:14.3f} "
      f"{muril_ref['diagnostic_by_query_type']['cohens_d']:15.3f}")
print(f"{'LaBSE':10s} {labse_ref['diagnostic_by_category']['cohens_d']:14.3f} "
      f"{labse_ref['diagnostic_by_query_type']['cohens_d']:15.3f}")
print(f"{'bge-m3':10s} {bge_diag_cat['cohens_d']:14.3f} {bge_diag_qtype['cohens_d']:15.3f}")

print(f"\nSame-category / different-category absolute means (bge-m3): "
      f"{bge_diag_cat['same_mean']:.4f} / {bge_diag_cat['diff_mean']:.4f}")
print(f"(Compare against the M3 report \u00a77.2 claim of ~0.404 median cosine for")
print(f" unrelated chunks under bge-m3, measured on a different dataset.)")


Loaded prior MuRIL/LaBSE numbers from: /content/drive/MyDrive/kcc_raw/final/kcc_embedding_comparison_summary.json

Model        d (Category)   d (QueryType)
MuRIL              -0.001           0.722
LaBSE               0.047           0.719
bge-m3              0.315           0.658

Same-category / different-category absolute means (bge-m3): 0.5672 / 0.5375
(Compare against the M3 report §7.2 claim of ~0.404 median cosine for
 unrelated chunks under bge-m3, measured on a different dataset.)


## Step 6: Build a Quick FAISS Index + Self-Retrieval Sanity

In [9]:
bge_index = faiss.IndexFlatIP(BGE_DIM)
bge_index.add(bge_embeddings.astype('float32'))
print(f"\u2705 bge-m3 index: {bge_index.ntotal:,} vectors, dim={bge_index.d}")

rng = np.random.default_rng(1)
sanity_idx = rng.choice(len(chunks), size=min(80, len(chunks)), replace=False)
hits = 0
for idx in sanity_idx:
    _, retrieved = bge_index.search(bge_embeddings[idx:idx+1].astype('float32'), k=1)
    if retrieved[0][0] == idx:
        hits += 1
print(f"Self-retrieval accuracy: {hits/len(sanity_idx):.1%} (expect ~100% \u2014 sanity check only)")


✅ bge-m3 index: 1,000 vectors, dim=1024
Self-retrieval accuracy: 100.0% (expect ~100% — sanity check only)


## Step 7: Quick Held-Out Generalization Check

Smaller than the full `06` run (100 queries instead of 200), same method:
held-out queries (approximated here by excluding exact text matches from
the sample) checked for category/crop match@5 against a random baseline.

In [10]:
FULL_CSV_PATH = f"{PROCESSED_PATH}kcc_cleaned_all_crops.csv"
if not Path(FULL_CSV_PATH).exists():
    print("\u26a0\ufe0f  Full cleaned CSV not found \u2014 skipping held-out eval, qualitative test only (Step 8).")
    RUN_HELD_OUT = False
else:
    RUN_HELD_OUT = True
    full_df = pd.read_csv(FULL_CSV_PATH)
    indexed_prefixes = set(t[:80] for t in texts)

    def qa_prefix(row):
        q = str(row.get('cleaned_query', row.get('QueryText', '')))
        a = str(row.get('cleaned_answer', row.get('KccAns', '')))
        return f"Question: {q}\nAnswer: {a}"[:80]

    full_df['_prefix'] = full_df.apply(qa_prefix, axis=1)
    held_out_df = full_df[~full_df['_prefix'].isin(indexed_prefixes)].copy()

    def stratified_df_sample(df, n_total, strat_col='Category', random_state=42):
        frames = []
        for cat, group in df.groupby(strat_col):
            share = len(group) / len(df)
            n_take = max(1, round(share * n_total))
            n_take = min(n_take, len(group))
            frames.append(group.sample(n_take, random_state=random_state))
        out = pd.concat(frames)
        return out.sample(min(n_total, len(out)), random_state=random_state).reset_index(drop=True)

    eval_df = stratified_df_sample(held_out_df, 100, strat_col='Category', random_state=42)
    print(f"Held-out eval set: {len(eval_df):,} queries")


Held-out eval set: 100 queries


In [11]:
if RUN_HELD_OUT:
    query_col = 'cleaned_query' if 'cleaned_query' in eval_df.columns else 'QueryText'
    K = 5

    def bge_search(query_text, k=K):
        q_vec = embed_bge([query_text], batch_size=1)[0:1].astype('float32')
        scores, ids = bge_index.search(q_vec, k)
        return [{"id": int(i), "score": float(s), "metadata": chunks[i]['metadata']}
                 for s, i in zip(scores[0], ids[0]) if i != -1]

    def random_search(query_text, k=K):
        rng_q = np.random.default_rng(hash(query_text) % (2**32))
        ids = rng_q.choice(len(chunks), size=k, replace=False)
        return [{"id": int(i), "metadata": chunks[i]['metadata']} for i in ids]

    bge_cat_hits, bge_crop_hits = [], []
    rand_cat_hits, rand_crop_hits = [], []
    latencies = []

    for _, row in eval_df.iterrows():
        q = str(row[query_col])
        if not q.strip():
            continue
        t0 = time.time()
        bge_results = bge_search(q)
        latencies.append((time.time() - t0) * 1000)
        rand_results = random_search(q)

        true_cat, true_crop = row.get('Category', 'unknown'), row.get('Crop', 'unknown')
        bge_cat_hits.append(any(r['metadata'].get('category') == true_cat for r in bge_results))
        bge_crop_hits.append(any(r['metadata'].get('crop') == true_crop for r in bge_results))
        rand_cat_hits.append(any(r['metadata'].get('category') == true_cat for r in rand_results))
        rand_crop_hits.append(any(r['metadata'].get('crop') == true_crop for r in rand_results))

    print(f"Held-Out Generalization (n={len(bge_cat_hits)}, k={K})")
    print("-" * 55)
    print(f"{'Method':10s} {'Category@5':>12s} {'Crop@5':>10s}")
    print(f"{'bge-m3':10s} {np.mean(bge_cat_hits):11.1%} {np.mean(bge_crop_hits):9.1%}")
    print(f"{'random':10s} {np.mean(rand_cat_hits):11.1%} {np.mean(rand_crop_hits):9.1%}")
    print(f"\nLift over random \u2014 category: {np.mean(bge_cat_hits)-np.mean(rand_cat_hits):+.1%}, "
          f"crop: {np.mean(bge_crop_hits)-np.mean(rand_crop_hits):+.1%}")
    print(f"Latency: mean={np.mean(latencies):.1f}ms, p50={np.percentile(latencies,50):.1f}ms")


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

...

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Held-Out Generalization (n=100, k=5)
-------------------------------------------------------
Method       Category@5     Crop@5
bge-m3           89.0%     81.0%
random           61.0%     19.0%

Lift over random — category: +28.0%, crop: +62.0%
Latency: mean=315.2ms, p50=293.5ms


## Step 8: Qualitative Check \u2014 Same 10 Farmer Queries as Before

Same test queries used for MuRIL/LaBSE, so you can eyeball bge-m3's
results against what you've already seen from the other two models.

In [12]:
TEST_QUERIES = [
    "wheat crop is turning yellow what to do",
    "gehu mein pila rog laga hai kya kare",
    "\u0917\u0947\u0939\u0942\u0902 \u0915\u0940 \u092b\u0938\u0932 \u092e\u0947\u0902 \u092a\u0940\u0932\u093e \u0930\u094b\u0917 \u0915\u094d\u092f\u093e \u0915\u0930\u0947\u0902",
    "best fertilizer for rice paddy",
    "PM Kisan yojana eligibility kaise check kare",
    "paddy pest control organic method",
    "mandi bhav for wheat today",
    "sugarcane disease red rot treatment",
    "how much water needed for maize crop",
    "onion price today up",
]

for q in TEST_QUERIES:
    q_vec = embed_bge([q], batch_size=1)[0:1].astype('float32')
    scores, ids = bge_index.search(q_vec, 3)
    print(f"\nQuery: {q}")
    for rank, (score, idx) in enumerate(zip(scores[0], ids[0]), start=1):
        if idx == -1:
            continue
        c = chunks[idx]
        print(f"  [{rank}] score={score:.3f} | crop={c['metadata'].get('crop')} | category={c['metadata'].get('category')}")
        print(f"       {c['text'][:120].replace(chr(10), ' ')}...")


Batches:   0%|          | 0/1 [00:00<?, ?it/s]


Query: wheat crop is turning yellow what to do
  [1] score=0.731 | crop=Wheat | category=Cereals
       Question: The leaves of the wheat crop are turning yellow and getting scorched, information about its control ? Answer: ...
  [2] score=0.682 | crop=Wheat | category=Cereals
       Question: Information about problem of yellowing in Wheat crop.? Answer: श्रीमान जी, NPK 19 19 19 10 ग्राम तथा सागरिका ज...
  [3] score=0.640 | crop=Wheat | category=Cereals
       Question: Wheat leaves are shriveled and pale yellow, information about it..? Answer: श्रीमान जी आपकी गेहूं की पत्तियां ...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]


Query: gehu mein pila rog laga hai kya kare
  [1] score=0.488 | crop=Paddy (Dhan) | category=Cereals
       Question: dhan me vikash kam ho raha hai ? Answer: srimaan ji aap dhan ki fasal me urea sagarika zink sulphate bataye ga...
  [2] score=0.456 | crop=Green Gram (Moong Bean/ Moong) | category=Pulses
       Question: MOONG KI PATTI ME CHED HO RAHE HAI KAUN SI DAWA DALE Answer: डायमेथोएट 30 EC 500 एम एल को 150 लीटर पानी में मि...
  [3] score=0.448 | crop=Brussils Sprouts | category=Vegetables
       Question: PARWAL ME JHULSA LAG RAHA HAI KYA DALE Answer: मेंकोंजेब 75 WP 500 ग्राम को 150 लीटर पानी में मिलाकर प्रति एकर...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

...


Query: sugarcane disease red rot treatment
  [1] score=0.646 | crop=Sugarcane (Noble Cane) | category=Sugar and Starch Crops
       Question: Information about control of red rot disease in sugarcane crops ? Answer: श्रीमान जी गन्ने की फसल में Thiophan...
  [2] score=0.635 | crop=Sugarcane (Noble Cane) | category=Sugar and Starch Crops
       Question: Information about control of stem rot(Wilt) disease in Sugarcane crop? Answer: श्रीमान जी,कार्बेन्डाजिम 50 WP ...
  [3] score=0.568 | crop=Paddy (Dhan) | category=Cereals
       Question: Information about Precaution of Red Rot Problem in Sugarcane Crop..? Answer: महोदय गन्ने में लाल सड़न रोग से ब...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]


Query: how much water needed for maize crop
  [1] score=0.659 | crop=Maize (Makka) | category=Millets
       Question: Information about nutrient management in Maize crop..? Answer: श्रीमान जी, मक्का की फसल में एनपीके (00 52 34) ...
  [2] score=0.652 | crop=Maize (Makka) | category=Millets
       Question: What to do to increase growth in Maize crop ..? Answer: श्री मान जी आप मक्का में NPK 18 18 18 1.25 kg एकर 180 ...
  [3] score=0.628 | crop=Maize (Makka) | category=Millets
       Question: Information about growth of maize crop.. ? Answer: महोदय फसल की ग्रोथ के लिए यूरिया 35 kg,सागरिका -10 kg एकर प...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]


Query: onion price today up
  [1] score=0.525 | crop=Onion | category=Vegetables
       Question: Provide about information of growth problem in Onion...? Answer: श्री मान जी आप प्याज में NPK 18 18 18 1.25 kg...
  [2] score=0.514 | crop=Onion | category=Vegetables
       Question: Give information about what to do for good production of onion ? Answer: श्रीमान जी प्याज के अच्छे उत्पादन के ...
  [3] score=0.513 | crop=Onion | category=Vegetables
       Question: Information about Variety in Onion Crop ? Answer: श्रीमान जी प्याज की प्रजातियां हैं एग्रीफाउंड लाइट रेड,कल्या...


## Step 9: Save Summary + Verdict

In [13]:
summary = {
    "model": MODEL_NAME,
    "dim": int(BGE_DIM),
    "n_chunks_tested": len(chunks),
    "embed_time_sec": round(bge_time, 1),
    "diagnostic_by_category": bge_diag_cat,
    "diagnostic_by_query_type": bge_diag_qtype,
    "comparison_reference": {"muril": muril_ref, "labse": labse_ref},
    "note": (
        "This is a quick test on a 1,000-chunk KCC sample, run to check whether the "
        "M3 report's \u00a77.2 bge-m3 anisotropy claim (measured on a different dataset) "
        "holds on KCC specifically. Not a full 5,000-chunk / 500-pair run like the "
        "MuRIL/LaBSE comparison \u2014 treat as a directional check, not a final number."
    ),
}
if RUN_HELD_OUT:
    summary["held_out_eval"] = {
        "n": len(bge_cat_hits), "k": K,
        "bge_category_match": float(np.mean(bge_cat_hits)),
        "bge_crop_match": float(np.mean(bge_crop_hits)),
        "random_category_match": float(np.mean(rand_cat_hits)),
        "random_crop_match": float(np.mean(rand_crop_hits)),
    }

out_path = f"{FINAL_PATH}kcc_bge_m3_quick_test_summary.json"
with open(out_path, 'w', encoding='utf-8') as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

print(json.dumps(summary, indent=2, ensure_ascii=False))
print(f"\nSaved: {out_path}")


{
  "model": "BAAI/bge-m3",
  "dim": 1024,
  "n_chunks_tested": 1000,
  "embed_time_sec": 551.4,
  "diagnostic_by_category": {
    "same_mean": 0.5671676993370056,
    "same_std": 0.10278801620006561,
    "diff_mean": 0.5375124216079712,
    "diff_std": 0.08456709235906601,
    "cohens_d": 0.31455549597740173
  },
  "diagnostic_by_query_type": {
    "same_mean": 0.5830148458480835,
    "same_std": 0.08507160097360611,
    "diff_mean": 0.52753084897995,
    "diff_std": 0.08325646817684174,
    "cohens_d": 0.6580985188484192
  },
  "comparison_reference": {
    "muril": {
      "model": "google/muril-base-cased",
      "dim": 768,
      "embed_time_sec": 2058.9,
      "self_retrieval": 1.0,
      "diagnostic_by_category": {
        "same_mean": 0.9970700740814209,
        "same_std": 0.0013326621847227216,
        "diff_mean": 0.9970710277557373,
        "diff_std": 0.001257210737094283,
        "cohens_d": -0.0007354153785854578
      },
      "diagnostic_by_query_type": {
        "same

---
## What To Do With This

- If bge-m3's Cohen's d and held-out lift here come out **higher** than
  LaBSE's, that supports switching \u2014 but re-run this at full scale
  (5,000 chunks, 500 pairs, like `05`/`06`) before finalizing, since this
  was intentionally a fast/small check.
- If bge-m3 looks **similar to or worse than** LaBSE on your KCC data,
  that's evidence the \u00a77.2 number in the M3 report doesn't transfer from
  your teammate's dataset to KCC, and the report should say so explicitly
  rather than imply one embedding choice fits both corpora.
- Either way, cite *this* notebook's numbers (measured on KCC) in the
  report instead of the unattributed 0.404 figure currently in \u00a77.2.
